# `semantic.v_beat_rate` — view

# Semantic — the dashboard views

Five views, each a join and a `GROUP BY` over Gold. Nothing is recalculated.

| view | grain | what the dashboard does with it |
|---|---|---|
| `v_beat_rate` | horizon | **the headline** — beat rate, risk and income, one row per horizon |
| `v_leaderboard` | ticker × horizon | best and worst, by return, income or risk — **SPY ranked in** |
| `v_beat_rate_by_cut` | cut × value × horizon | the three cuts: management group, sole/multi, sector |
| `v_measure` | ticker × horizon × measure | the same facts on **three measures**, so one filter switches the page |
| `v_measure_summary` | horizon × measure | **the KPI row** — index %, best trust %, index rank |

**The leaderboard never feeds the beat rate.** Rank to a top ten and *then* ask "what share
beat the index?" and the answer is always 100%, because you picked them for winning. The
beat rate reads the whole fact; the leaderboard is display only.

A view is its own definition, so there is no load step and no etl task to pair with this one.

In [0]:
-- THE HEADLINE. Beat rate by horizon. The universe is the trusts still listed:
-- delisted ones were cut at Silver, so there is nothing to filter out here.
CREATE OR REPLACE VIEW `index-vs-trust-pipeline`.semantic.v_beat_rate
COMMENT 'Beat rate, risk and income by horizon, across the listed trust universe'
AS
WITH spy AS (
  SELECT horizon_years, volatility AS spy_volatility, total_return AS spy_return,
         income_return AS spy_income, price_return AS spy_price
  FROM `index-vs-trust-pipeline`.semantic.v_horizon_performance
  WHERE ticker = 'SPY'
),
trusts AS (
  SELECT f.horizon_years, f.ticker, f.total_return, f.volatility, f.income_return,
         f.risk_adjusted_return, f.beat_index, f.out_grew_index
  FROM `index-vs-trust-pipeline`.semantic.v_horizon_performance f
  JOIN `index-vs-trust-pipeline`.gold.dim_ticker d
    ON d.ticker_key = f.ticker_key AND d.entity_type = 'Trust'
)
SELECT t.horizon_years,
       COUNT(*)                                                      AS trusts,
       SUM(CASE WHEN t.beat_index THEN 1 ELSE 0 END)                 AS beat_count,
       ROUND(100.0 * SUM(CASE WHEN t.beat_index THEN 1 ELSE 0 END)
             / COUNT(*), 1)                                          AS beat_rate_pct,
       ROUND(100 * PERCENTILE_APPROX(t.total_return, 0.5), 1)        AS median_return_pct,
       ROUND(100 * MAX(s.spy_return), 1)                             AS index_return_pct,
       ROUND(100 * PERCENTILE_APPROX(t.volatility, 0.5), 1)          AS median_volatility_pct,
       ROUND(100 * MAX(s.spy_volatility), 1)                         AS index_volatility_pct,
       ROUND(100.0 * SUM(CASE WHEN t.volatility < s.spy_volatility
                              THEN 1 ELSE 0 END) / COUNT(*), 1)      AS calmer_than_index_pct,
       -- The only unambiguous win: beat it AND took less risk doing so.
       ROUND(100.0 * SUM(CASE WHEN t.beat_index AND t.volatility < s.spy_volatility
                              THEN 1 ELSE 0 END) / COUNT(*), 1)      AS beat_and_calmer_pct,
       -- The counterweight. Income is cumulative over the window, not an annual yield.
       SUM(CASE WHEN t.income_return > s.spy_income THEN 1 ELSE 0 END)
                                                                     AS out_yielded_count,
       ROUND(100.0 * SUM(CASE WHEN t.income_return > s.spy_income
                              THEN 1 ELSE 0 END) / COUNT(*), 1)      AS out_yielded_pct,
       ROUND(100 * PERCENTILE_APPROX(t.income_return, 0.5), 1)       AS median_income_pct,
       ROUND(100 * MAX(s.spy_income), 1)                             AS index_income_pct,
       -- Growth alone, before dividends. A different question from beat_rate_pct.
       SUM(CASE WHEN t.out_grew_index THEN 1 ELSE 0 END)              AS out_grew_count,
       ROUND(100.0 * SUM(CASE WHEN t.out_grew_index THEN 1 ELSE 0 END)
             / COUNT(*), 1)                                           AS out_grew_pct,
       ROUND(100 * MAX(s.spy_price), 1)                               AS index_price_return_pct
FROM trusts t
JOIN spy s ON s.horizon_years = t.horizon_years
GROUP BY t.horizon_years;

## Verification

Expected: **5 rows**, one per horizon.

In [0]:
SELECT COUNT(*) AS rows
FROM `index-vs-trust-pipeline`.semantic.v_beat_rate;